In [5]:
import os
import re
import json
import pandas as pd
from openai import OpenAI
import dotenv

dotenv.load_dotenv()

TYPHOON_API_KEY = os.getenv("TYPHOON_API_KEY")

client = OpenAI(api_key=TYPHOON_API_KEY, base_url="https://api.opentyphoon.ai/v1")

SYSTEM_PROMPT = """คุณเป็นผู้เชี่ยวชาญถอดความหมายและสรุปโครงสร้าง “ประกาศขายอสังหาริมทรัพย์ภาษาไทย” สำหรับนำไปสร้างตารางสรุป 5 คอลัมน์หลัก คือ ประเภท, ราคา, ทำเล, ขนาด, Link

หน้าที่หลักของคุณคือ:
1. วิเคราะห์ข้อความในประกาศอย่างละเอียด
2. ดึงรายละเอียดโครงสร้างให้ครบถ้วนที่สุดลงใน extracted object โดยเน้นความแม่นยำของ "ทำเล" และ "ขนาด"

การดึงข้อมูลโครงสร้างใน extracted:
- property_type:
  - ระบุประเภทอสังหาให้ชัดเจน เช่น “บ้านเดี่ยว”, “บ้านแฝด”, “ทาวน์เฮ้าส์”, “คอนโด”, “อาคารพาณิชย์”, “ที่ดิน”, “วิลล่า”, “อพาร์ตเมนต์”, “โฮมออฟฟิศ”, หรือ “อื่นๆ” ที่ใกล้เคียงที่สุด
- price_text และ price_value_thb:
  - อ่านรูปแบบราคาไทยทุกแบบ เช่น “ราคา 17,900,000 บาท”, “ราคาเพียง 1.6 ล้านบาท”, “ขายเพียง 2,200,000 บาท”, “2.5ล.”
  - เก็บข้อความราคาเต็มดั้งเดิมไว้ใน price_text
  - แปลงเป็นจำนวนเงินหน่วยบาทใน price_value_thb (float) ถ้าไม่มีราคาขายให้ใส่ null
- location_text (จะใช้เป็นคอลัมน์ “ทำเล”):
  - **สำคัญมาก:** ดึงข้อมูลระบุตำแหน่งทั้งหมดให้ละเอียดที่สุดเท่าที่จะหาได้
  - รวมชื่อโครงการ, หมู่บ้าน, คอนโด, เลขที่บ้าน, ซอย, ถนน, แขวง/ตำบล, เขต/อำเภอ, จังหวัด
  - รวมสถานที่ใกล้เคียง (Landmarks) ทั้งหมด เช่น ใกล้ห้าง, ใกล้มหาวิทยาลัย, ใกล้รถไฟฟ้า, ย่านธุรกิจ
  - นำข้อมูลทั้งหมดมาต่อกันคั่นด้วย “ | ”
- size_text (จะใช้เป็นคอลัมน์ “ขนาด”):
  - ดึงข้อความที่เกี่ยวกับขนาดพื้นที่ดิน (ตร.วา, ไร่, งาน) และพื้นที่ใช้สอย (ตร.ม.)
  - รวมข้อมูลจำนวนชั้น (ถ้ามี)
  - นำข้อมูลทั้งหมดมาต่อกันคั่นด้วย “ | ”
- bedrooms และ bathrooms:
  - แปลงจำนวนห้องนอน/ห้องน้ำเป็นตัวเลข int (ถ้ามี) เพื่อนำไปประกอบในช่องขนาด

รูปแบบคำตอบ:
- ตอบเป็น JSON เดียวบรรทัดเดียวเท่านั้น
- ห้ามใช้ code block
- ห้ามมีข้อความอื่นนอกเหนือ JSON
- ต้องเป็นไปตามสคีมา:
{"extracted": {"property_type": "string", "bedrooms": int|null, "bathrooms": int|null, "size_text": "string", "location_text": "string", "price_text": "string", "price_value_thb": float|null}}"""

PROPERTY_PATTERNS = [
    r"บ้านเดี่ยว",
    r"บ้านแฝด",
    r"บ้าน(?!พักคนงาน)",
    r"คอนโด",
    r"ทาวน์",
    r"ทาวน์โฮม",
    r"อาคารพาณิชย์",
    r"ตึกแถว",
    r"ที่ดิน",
    r"โกดัง",
    r"โรงงาน",
    r"อพาร์ตเมนต์",
    r"แมนชั่น",
    r"วิลลา",
    r"คฤหาสน์",
    r"โฮมออฟฟิศ",
    r"สำนักงาน",
    r"ออฟฟิศ",
    r"\bcondo\b",
    r"\bhouse\b",
    r"\btownhouse\b",
    r"\bland\b",
    r"\bwarehouse\b",
    r"\bapartment\b",
    r"\boffice\b",
    r"\bvilla\b",
    r"\bmansion\b",
    r"\bpenthouse\b",
]

RENT_OR_IRRELEVANT_PATTERNS = [
    r"ให้เช่า",
    r"\bเช่า\b",
    r"ปล่อยเช่า",
    r"เช่ารายวัน",
    r"เช่ารายเดือน",
    r"ค่าเช่า",
    r"ประกันห้อง",
    r"มัดจำ",
]

prop_regex = [re.compile(p, flags=re.IGNORECASE) for p in PROPERTY_PATTERNS]
rent_regex = [re.compile(p, flags=re.IGNORECASE) for p in RENT_OR_IRRELEVANT_PATTERNS]


def normalize_text(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = re.sub(r"\s+", " ", s.replace("\u200b", " ")).strip()
    s = s.replace("ดูน้อยลง", "")
    return s


def is_real_estate(title: str, details: str, desc: str) -> bool:
    t = normalize_text((title or "") + " " + (details or "") + " " + (desc or ""))
    return any(r.search(t) for r in prop_regex)


def is_sale_post(title: str, details: str, desc: str) -> bool:
    t = normalize_text((title or "") + " " + (details or "") + " " + (desc or ""))
    if any(r.search(t) for r in rent_regex):
        return False
    return True


def clamp_text(s: str, n: int = 3500) -> str:
    if not isinstance(s, str):
        return ""
    return s[:n]


def build_user_message(row):
    msg = (
        "URL: " + clamp_text(row.get("URL", ""))
        + "\nTITLE: " + clamp_text(row.get("Title", ""))
        + "\nPRICE: " + clamp_text(row.get("Price", ""))
        + "\nDETAILS: " + clamp_text(row.get("Property_Details", ""))
        + "\nDESCRIPTION:\n" + clamp_text(row.get("Description", ""))
    )
    return msg


def extract_json_blob(s: str) -> str:
    s = s.strip().replace("\u200b", "")
    s = re.sub(r"^```(?:json)?\s*|\s*```$", "", s, flags=re.IGNORECASE).strip()
    i = s.find("{")
    j = s.rfind("}")
    if i != -1 and j != -1 and j >= i:
        return s[i : j + 1]
    return s


def parse_extracted(s: str) -> dict:
    d = {}
    jb = extract_json_blob(s)
    m = re.search(r'"?extracted"?\s*:\s*\{(.*)\}', jb, re.S)
    block = m.group(1) if m else ""

    def grab_str(k):
        m2 = re.search(rf'"?{k}"?\s*:\s*"([^"]*)"', block)
        if m2:
            return m2.group(1)
        m3 = re.search(rf"'{k}'\s*:\s*'([^']*)'", block)
        return m3.group(1) if m3 else None

    def grab_int_or_null(k):
        m2 = re.search(rf'"?{k}"?\s*:\s*(\d+|null|None)', block, re.I)
        if not m2:
            return None
        v = m2.group(1)
        if v.lower() in ("null", "none"):
            return None
        return int(v)

    def grab_float_or_null(k):
        m2 = re.search(
            rf'"?{k}"?\s*:\s*(null|None|[0-9]+(?:\.[0-9]+)?)', block, re.I
        )
        if not m2:
            return None
        v = m2.group(1)
        if v.lower() in ("null", "none"):
            return None
        return float(v)

    d["property_type"] = grab_str("property_type")
    d["bedrooms"] = grab_int_or_null("bedrooms")
    d["bathrooms"] = grab_int_or_null("bathrooms")
    d["size_text"] = grab_str("size_text")
    d["location_text"] = grab_str("location_text")
    d["price_text"] = grab_str("price_text")
    d["price_value_thb"] = grab_float_or_null("price_value_thb")
    return d


def call_typhoon_extract(json_input_text: str) -> str:
    r = client.chat.completions.create(
        model="typhoon-v2.5-30b-a3b-instruct",
        temperature=0.1,
        max_tokens=8192,
        top_p=0.96,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": json_input_text},
        ],
    )
    return r.choices[0].message.content


def format_price_thb(n):
    if n is None:
        return "-"
    if n >= 1_000_000:
        v = round(n / 1_000_000, 2)
        s = f"{v}".rstrip("0").rstrip(".")
        return f"{s} ล้านบาท"
    if n >= 1_000:
        v = round(n / 1_000, 0)
        return f"{int(v)} พันบาท"
    return f"{n} บาท"


def build_row(row, extracted: dict) -> dict:
    ptxt = extracted.get("price_text")
    pval = extracted.get("price_value_thb")
    if isinstance(pval, str):
        cleaned = pval.replace(",", "").strip()
        if cleaned == "" or cleaned.lower() in ("nan", "null", "none"):
            pval = None
        elif re.fullmatch(r"-?\d+(\.\d+)?", cleaned):
            pval = float(cleaned)
        else:
            pval = None
    price_txt = (
        ptxt
        if (isinstance(ptxt, str) and ptxt.strip() != "")
        else (format_price_thb(pval) if pval is not None else "-")
    )
    ptype = extracted.get("property_type") or "-"
    size_txt = extracted.get("size_text") or "-"
    beds = extracted.get("bedrooms")
    baths = extracted.get("bathrooms")
    bb_parts = []
    if beds is not None:
        bb_parts.append(f"{beds} นอน")
    if baths is not None:
        bb_parts.append(f"{baths} น้ำ")
    
    bb = " ".join(bb_parts)
    if bb:
        size_field = (size_txt + " | " + bb).strip()
    else:
        size_field = size_txt
        
    loc = extracted.get("location_text") or "-"
    url = row.get("URL", "-")
    
    # Format: ประเภท, ราคา, ทำเล, ขนาด, Link
    return {
        "ประเภท": ptype,
        "ราคา": price_txt,
        "ทำเล": loc,
        "ขนาด": size_field,
        "Link": url,
    }


def process_and_save(input_path: str, output_path: str):
    print("load_csv:", input_path)
    df = pd.read_csv(input_path)
    print("shape:", df.shape)
    print("columns:", list(df.columns))
    
    # Map columns to standard names
    if "Post_URL" in df.columns:
        df["URL"] = df["Post_URL"]
    if "Full_Post_Content" in df.columns:
        df["Description"] = df["Full_Post_Content"]
    for c in ["Title", "Property_Details", "Price"]:
        if c not in df.columns:
            df[c] = ""
            
    rows = []
    print("start_iterate")
    
    for i, row in df.iterrows():
        title = normalize_text(row.get("Title", ""))
        details = normalize_text(row.get("Property_Details", ""))
        desc = normalize_text(row.get("Description", ""))
        
        # Filter Logic
        if not is_real_estate(title, details, desc):
            if i % 50 == 0:
                print("skip_non_real_estate_index:", i)
            continue
        if not is_sale_post(title, details, desc):
            if i % 50 == 0:
                print("skip_non_sale_index:", i)
            continue
            
        # Build Message & Call AI
        user_msg = build_user_message({
            "URL": row.get("URL", ""),
            "Title": title,
            "Price": row.get("Price", ""),
            "Property_Details": details,
            "Description": desc,
        })
        
        print("typhoon_call_index:", i)
        raw = call_typhoon_extract(user_msg)
        blob = extract_json_blob(raw)
        
        # Extract & Build Row
        extracted = parse_extracted(blob)
        out = build_row(row, extracted)
        
        rows.append(out)
        print("processed_index:", i, "type:", out["ประเภท"], "loc:", out["ทำเล"][:30])

    print("kept_rows:", len(rows))
    out_df = pd.DataFrame(
        rows, columns=["ประเภท", "ราคา", "ทำเล", "ขนาด", "Link"]
    )
    
    output_dir = os.path.dirname(output_path)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)

    out_df.to_excel(output_path, index=False)
    print("saved_excel:", output_path)


if __name__ == "__main__":
    input_path = r"Scraping\scraped_full_content.csv"
    output_path = r"Scraping\owner_score.xlsx"
    process_and_save(input_path, output_path)

load_csv: Scraping\scraped_full_content.csv
shape: (1028, 2)
columns: ['Post_URL', 'Full_Post_Content']
start_iterate
typhoon_call_index: 0
processed_index: 0 type: บ้านเดี่ยว loc: ท่ารั้ว | สันปูเลย | เชียงใหม่
typhoon_call_index: 2
processed_index: 2 type: ที่ดิน loc: ม.6 ต.ออนเหนือ อ.สันกำแพง จ.เช
typhoon_call_index: 4
processed_index: 4 type: บ้านเดี่ยว loc: หมู่บ้านบ้านวังน้ำริน | ต.สันผ
typhoon_call_index: 5
processed_index: 5 type: บ้านเดี่ยว loc: อำเภอเมือง เชียงใหม่ | ตำบลฟ้า
typhoon_call_index: 6
processed_index: 6 type: ที่ดิน loc: บ้านสันกับตองใต้ ซ.2 ต.สารภี อ
typhoon_call_index: 7
processed_index: 7 type: บ้านเดี่ยว loc: โครงการศิริภัสสร 3 | เทศบาลตำบ
typhoon_call_index: 8
processed_index: 8 type: คอนโด loc: ศรีอนันต์ คอนโด | ติดถนนซุปเปอ
typhoon_call_index: 10
processed_index: 10 type: ที่ดิน loc: ต.ห้วยแก้ว แม่ออน | ใกล้แหล่งท
typhoon_call_index: 12
processed_index: 12 type: บ้านเดี่ยว loc: อ.ดอยหล่อ จ.เชียงใหม่ | ใกล้โร
typhoon_call_index: 13
processed_index: 13 type: 

In [1]:
import os
import re
import json
import pandas as pd
from openai import OpenAI
import dotenv
from openpyxl import load_workbook, Workbook

dotenv.load_dotenv()

TYPHOON_API_KEY = os.getenv("TYPHOON_API_KEY")

client = OpenAI(api_key=TYPHOON_API_KEY, base_url="https://api.opentyphoon.ai/v1")

SYSTEM_PROMPT = """คุณเป็นผู้เชี่ยวชาญถอดความหมายประกาศอสังหาริมทรัพย์
หน้าที่: วิเคราะห์ข้อความและดึงข้อมูลใส่ JSON สำหรับ extracted

field ที่ต้องการ:
- property_type: ประเภทอสังหา เช่น บ้านเดี่ยว, คอนโด, ที่ดิน
- listing_type: ประเภทประกาศ ให้ระบุว่า "ขาย", "เช่า", หรือ "ขาย/เช่า"
- price_text: ข้อความราคาเต็ม
- price_value_thb: ราคาเป็นตัวเลข (บาท)
- primary_location: **สำคัญ** ให้เลือกชื่อ "อำเภอ" หรือ "ย่านหลัก" เพียง 1 ชื่อที่เด่นชัดที่สุดเพื่อใช้เป็นตัวกรอง (เช่น "เมืองเชียงใหม่", "หางดง", "นิมมาน")
- location_tags: ระบุทำเล ย่าน ถนน สถานที่ใกล้เคียง ทั้งหมดที่มี คั่นด้วยเครื่องหมายจุลภาค (,)
- size_text: ขนาดพื้นที่และพื้นที่ใช้สอย คั่นด้วยจุลภาค (,)
- bedrooms: จำนวนห้องนอน (int)
- bathrooms: จำนวนห้องน้ำ (int)
- post_date: วันที่ลงประกาศหรืออัพเดทล่าสุดที่ระบุในข้อความ (เช่น 12 ม.ค. 67, 2024-01-01) ถ้าไม่พบให้ใส่ null

รูปแบบคำตอบ (JSON Only):
{"extracted": {"property_type": "string", "listing_type": "string", "primary_location": "string", "location_tags": "string", "bedrooms": int|null, "bathrooms": int|null, "size_text": "string", "price_text": "string", "price_value_thb": float|null, "post_date": "string|null"}}"""

PROPERTY_PATTERNS = [
    r"บ้าน", r"คอนโด", r"ทาวน์", r"อาคารพาณิชย์", r"ตึกแถว", r"ที่ดิน",
    r"โกดัง", r"โรงงาน", r"อพาร์ตเมนต์", r"แมนชั่น", r"วิลลา", r"โฮมออฟฟิศ",
    r"หอพัก", r"ห้องเช่า", r"ปล่อยเช่า", r"ให้เช่า", r"ขาย"
]

prop_regex = [re.compile(p, flags=re.IGNORECASE) for p in PROPERTY_PATTERNS]

def normalize_text(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = re.sub(r"\s+", " ", s.replace("\u200b", " ")).strip()
    s = s.replace("ดูน้อยลง", "")
    return s

def is_relevant_post(title: str, details: str, desc: str) -> bool:
    t = normalize_text((title or "") + " " + (details or "") + " " + (desc or ""))
    return any(r.search(t) for r in prop_regex)

def clamp_text(s: str, n: int = 3500) -> str:
    if not isinstance(s, str):
        return ""
    return s[:n]

def build_user_message(row):
    msg = (
        "URL: " + clamp_text(row.get("URL", ""))
        + "\nTITLE: " + clamp_text(row.get("Title", ""))
        + "\nPRICE: " + clamp_text(row.get("Price", ""))
        + "\nDETAILS: " + clamp_text(row.get("Property_Details", ""))
        + "\nDESCRIPTION:\n" + clamp_text(row.get("Description", ""))
        + "\nPOST_DATE_RAW: " + clamp_text(str(row.get("Date", "")))
    )
    return msg

def extract_json_blob(s: str) -> str:
    s = s.strip().replace("\u200b", "")
    s = re.sub(r"^```(?:json)?\s*|\s*```$", "", s, flags=re.IGNORECASE).strip()
    i = s.find("{")
    j = s.rfind("}")
    if i != -1 and j != -1 and j >= i:
        return s[i : j + 1]
    return s

def parse_extracted(s: str) -> dict:
    d = {}
    jb = extract_json_blob(s)
    try:
        data = json.loads(jb)
        if "extracted" in data:
            data = data["extracted"]
        d = data
    except json.JSONDecodeError:
        pass
    
    return {
        "property_type": d.get("property_type"),
        "listing_type": d.get("listing_type"),
        "primary_location": d.get("primary_location"),
        "location_tags": d.get("location_tags"),
        "bedrooms": d.get("bedrooms"),
        "bathrooms": d.get("bathrooms"),
        "size_text": d.get("size_text"),
        "price_text": d.get("price_text"),
        "price_value_thb": d.get("price_value_thb"),
        "post_date": d.get("post_date")
    }

def call_typhoon_extract(json_input_text: str) -> str:
    r = client.chat.completions.create(
        model="typhoon-v2.5-30b-a3b-instruct",
        temperature=0.1,
        max_tokens=8192,
        top_p=0.96,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": json_input_text},
        ],
    )
    return r.choices[0].message.content

def format_price_thb(n):
    if n is None:
        return "-"
    if n >= 1_000_000:
        v = round(n / 1_000_000, 2)
        s = f"{v}".rstrip("0").rstrip(".")
        return f"{s} ล้านบาท"
    if n >= 1_000:
        v = round(n / 1_000, 0)
        return f"{int(v)} พันบาท"
    return f"{n} บาท"

def build_row(row, extracted: dict) -> dict:
    ptxt = extracted.get("price_text")
    pval = extracted.get("price_value_thb")
    
    price_txt = ptxt if (isinstance(ptxt, str) and len(ptxt) > 2) else format_price_thb(pval)
    
    ptype = extracted.get("property_type") or "-"
    ltype = extracted.get("listing_type") or "-"
    prim_loc = extracted.get("primary_location") or "-"
    loc_tags = extracted.get("location_tags") or "-"
    size_txt = extracted.get("size_text") or "-"
    beds = extracted.get("bedrooms")
    baths = extracted.get("bathrooms")
    
    # Priority: Date from CSV > Date from LLM
    csv_date = str(row.get("Date", "")).strip()
    llm_date = extracted.get("post_date") or ""
    final_date = csv_date if (csv_date and csv_date.lower() != "nan") else llm_date
    
    bb_parts = []
    if beds is not None and str(beds).isdigit(): bb_parts.append(f"{beds} นอน")
    if baths is not None and str(baths).isdigit(): bb_parts.append(f"{baths} น้ำ")
    
    bb = " ".join(bb_parts)
    if bb:
        size_field = (size_txt + ", " + bb).strip()
    else:
        size_field = size_txt
        
    url = row.get("URL", "-")
    
    return {
        "ประเภทประกาศ": ltype,
        "ประเภททรัพย์": ptype,
        "ราคา": price_txt,
        "ทำเลหลัก (Filter)": prim_loc,
        "รายละเอียดทำเล": loc_tags,
        "ขนาด": size_field,
        "วันที่": final_date,
        "Link": url,
    }

def append_to_excel(file_path, data_dict):
    headers = ["ประเภทประกาศ", "ประเภททรัพย์", "ราคา", "ทำเลหลัก (Filter)", "รายละเอียดทำเล", "ขนาด", "วันที่", "Link"]
    values = [data_dict.get(h, "") for h in headers]
    
    if not os.path.exists(file_path):
        wb = Workbook()
        ws = wb.active
        ws.append(headers)
        ws.append(values)
        wb.save(file_path)
    else:
        wb = load_workbook(file_path)
        ws = wb.active
        ws.append(values)
        wb.save(file_path)

def process_and_save(input_path: str, output_path: str):
    print("load_csv:", input_path)
    df = pd.read_csv(input_path)
    print("shape:", df.shape)
    
    if "Post_URL" in df.columns: df["URL"] = df["Post_URL"]
    if "Full_Post_Content" in df.columns: df["Description"] = df["Full_Post_Content"]
    # Check if 'Date' exists from scraper, if not create empty
    for c in ["Title", "Property_Details", "Price", "Date"]:
        if c not in df.columns: df[c] = ""
            
    output_dir = os.path.dirname(output_path)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)

    print("start_iterate")
    
    count = 0
    for i, row in df.iterrows():
        title = normalize_text(row.get("Title", ""))
        details = normalize_text(row.get("Property_Details", ""))
        desc = normalize_text(row.get("Description", ""))
        
        # กรองเฉพาะโพสต์อสังหาฯ แต่รับทั้ง ขายและเช่า
        if not is_relevant_post(title, details, desc):
            if i % 50 == 0: print(f"skip_index: {i}")
            continue
            
        user_msg = build_user_message({
            "URL": row.get("URL", ""),
            "Title": title,
            "Price": row.get("Price", ""),
            "Property_Details": details,
            "Description": desc,
            "Date": row.get("Date", "")
        })
        
        print(f"processing_index: {i}")
        
        raw = call_typhoon_extract(user_msg)
        blob = extract_json_blob(raw)
        extracted = parse_extracted(blob)
        out = build_row(row, extracted)
        
        append_to_excel(output_path, out)
        count += 1
        print(f"saved: {out['ประเภทประกาศ']} {out['ทำเลหลัก (Filter)']} | {out['วันที่']}")

    print(f"Done. Processed {count} rows. Saved to: {output_path}")

if __name__ == "__main__":
    input_path = r"CSV_file/scraped_full_content.csv"
    output_path = r"test.xlsx"
    process_and_save(input_path, output_path)

ModuleNotFoundError: No module named 'openpyxl'

In [4]:
import os
import re
import json
import pandas as pd
from openai import OpenAI
import dotenv

dotenv.load_dotenv()

TYPHOON_API_KEY = os.getenv("TYPHOON_API_KEY")
OUTPUT_EXCEL_FILE = os.getenv("OUTPUT_EXCEL_FILE", r"C:\Users\kongl\Documents\GitHub\Real-Estate Listing Aggregator System\CSV_file\fazwaz_classified_locations.xlsx")

client = OpenAI(api_key=TYPHOON_API_KEY, base_url="https://api.opentyphoon.ai/v1")

SYSTEM_PROMPT = """คุณคือ AI Data Extraction Engine ระดับสูงสำหรับวงการอสังหาริมทรัพย์เชียงใหม่ ภารกิจของคุณคือการแปลง Text ประกาศที่ไม่มีโครงสร้าง ให้กลายเป็น Structured JSON Object ที่มีความแม่นยำสูงสุด

**# MISSION CRITICAL RULES:**
1. FIELD-BY-FIELD EXTRACTION: สำหรับแต่ละ Key ใน "extracted" object คุณต้องค้นหาค่าที่ตรงกันใน Text หากไม่พบค่าสำหรับ Key ใด Key หนึ่ง ให้ใช้ `null` สำหรับ Key นั้นโดยเฉพาะ ห้ามข้าม Field อื่น
2. NO SUMMARIZATION: ห้ามสรุปหรือย่อข้อมูลเด็ดขาด ให้สกัดค่าตามจริงที่ปรากฏ

**# การสกัดข้อมูลทำเล (Location Extraction) - สำคัญที่สุด:**
คุณต้องสกัดข้อมูลทำเลลงใน `location` object ซึ่งมี 2 field:
1. `amphoe_chiangmai`: **(Classification Task)** ให้คุณระบุ "อำเภอ" ของทรัพย์สิน โดยต้องเลือกคำตอบจาก List ต่อไปนี้เท่านั้น: ["เมืองเชียงใหม่", "แม่ริม", "สันทราย", "สารภี", "หางดง", "ดอยสะเก็ด", "สันกำแพง", "สันป่าตอง", "เชียงดาว", "ฝาง", "แม่แจ่ม", "ฮอด", "แม่แตง", "พร้าว", "จอมทอง", "แม่อาย", "กัลยาณิวัฒนา", "อื่นๆ"] หากไม่พบ ให้ใช้ "อื่นๆ"
2. `full_location_text`: **(Extraction Task)** ให้คุณรวบรวม "ทุกข้อความ" ที่เกี่ยวกับทำเลทั้งหมด โดยเฉพาะอย่างยิ่ง **"ชื่อโครงการ"**, **"ชื่อหมู่บ้าน"**, **"ชื่อคอนโด"** รวมถึง บ้านเลขที่, ซอย, ถนน, ตำบล, และ "สถานที่ใกล้เคียงทั้งหมด" มารวมกันเป็น String เดียวโดยใช้ " | " คั่น

**# EXTRACTION SCHEMA & INSTRUCTIONS:**
สกัดข้อมูลตาม Schema นี้อย่างเคร่งครัด:
    * `property_type` (string): ประเภทหลัก เช่น "บ้านเดี่ยว", "คอนโด", "ที่ดิน"
    * `bedrooms` (int|null): จำนวนห้องนอน
    * `bathrooms` (int|null): จำนวนห้องน้ำ
    * `land_area_sqw` (float|null): ขนาดที่ดินในหน่วย "ตารางวา" หรือ "ตร.ว."
    * `usable_area_sqm` (float|null): พื้นที่ใช้สอยในหน่วย "ตารางเมตร" หรือ "ตร.ม."
    * `location` (object): object ที่มี `amphoe_chiangmai` และ `full_location_text`
    * `price_text` (string|null): ข้อความราคาดิบ
    * `price_value_thb` (float|null): ราคาที่แปลงเป็นตัวเลขแล้ว

**# OUTPUT FORMAT (JSON ONLY):**
ตอบเป็น JSON เดียวบรรทัดเดียวเท่านั้น ห้ามมีโค้ดบล็อคหรือข้อความอื่นนอกเหนือ JSON ตามสคีมาและตัวอย่างนี้:
{"extracted": {"property_type": "บ้านเดี่ยว", "bedrooms": 3, "bathrooms": 3, "land_area_sqw": 56.0, "usable_area_sqm": 122.0, "location": {"amphoe_chiangmai": "หางดง", "full_location_text": "โครงการกาญจน์กนกวิลล์ 10 | หมู่บ้านวังตาล | สันผักหวาน, หางดง, เชียงใหม่ | ใกล้: แม็คโคร หางดง, บิ๊กซี หางดง, สนามบินเชียงใหม่"}, "price_text": "฿3,300,000", "price_value_thb": 3300000.0}}"""

PROPERTY_PATTERNS = [
    re.compile(p, flags=re.IGNORECASE) for p in [
        r"บ้านเดี่ยว", r"บ้านแฝด", r"บ้าน", r"คอนโด", r"ทาวน์โฮม", r"ทาวน์เฮ้าส์",
        r"อาคารพาณิชย์", r"ตึกแถว", r"ที่ดิน", r"โกดัง", r"โรงงาน", r"อพาร์ตเมนต์",
        r"วิลล่า", r"โฮมออฟฟิศ", r"\bcondo\b", r"\bhouse\b", r"\bland\b"
    ]
]

def normalize_text(s: str) -> str:
    if not isinstance(s, str):
        return ""
    return re.sub(r"\s+", " ", s.replace("\u200b", " ").replace("ดูน้อยลง", "")).strip()

def is_relevant_post(full_text: str) -> bool:
    if not any(r.search(full_text) for r in PROPERTY_PATTERNS):
        return False
    return True

def clamp_text(s: str, n: int = 4000) -> str:
    if not isinstance(s, str):
        return ""
    return s[:n]

def build_user_message(row):
    return " ".join(filter(None, [
        row.get("Title", ""),
        row.get("Price", ""),
        row.get("Property_Details", ""),
        row.get("Description", "")
    ]))

def extract_json_blob(s: str) -> dict:
    s = s.strip()
    match = re.search(r"\{.*\}", s, re.DOTALL)
    if match:
        return json.loads(match.group(0))
    return {}

def call_typhoon_analyzer(user_message: str) -> dict:
    completion = client.chat.completions.create(
        model="typhoon-v2.5-30b-a3b-instruct",
        temperature=0.05,
        max_tokens=2048,
        top_p=1.0,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": clamp_text(user_message)}
        ],
    )
    content = completion.choices[0].message.content
    return extract_json_blob(content)

def build_row(row_data: dict, llm_result: dict) -> dict:
    extracted = llm_result.get("extracted", {})
    
    price_val = extracted.get("price_value_thb")
    price_text = extracted.get("price_text") or (f"{price_val/1_000_000:.2f} ล้านบาท".rstrip('0').rstrip('.') if price_val and price_val >= 1_000_000 else f"{price_val:,.0f} บาท" if price_val else "-")

    size_parts = []
    if extracted.get("land_area_sqw") is not None: size_parts.append(f"{extracted['land_area_sqw']} ตร.ว.")
    if extracted.get("usable_area_sqm") is not None: size_parts.append(f"{extracted['usable_area_sqm']} ตร.ม.")
    if extracted.get("bedrooms") is not None: size_parts.append(f"{extracted['bedrooms']} นอน")
    if extracted.get("bathrooms") is not None: size_parts.append(f"{extracted['bathrooms']} น้ำ")
    size_field = " | ".join(filter(None, size_parts)) or "-"

    location_obj = extracted.get("location", {})
    amphoe = location_obj.get("amphoe_chiangmai") or "N/A"
    full_location = location_obj.get("full_location_text") or "-"
    
    return {
        "ประเภท": extracted.get("property_type") or "-",
        "ราคา": price_text,
        "อำเภอ": amphoe,
        "ทำเล": full_location,
        "ขนาด": size_field,
        "Link": row_data.get("URL", "-")
    }

def process_and_save(input_path: str, output_path: str):
    print(f"Loading CSV from: {input_path}")
    df = pd.read_csv(input_path)
    print(f"Loaded {len(df)} rows.")
    
    if "Post_URL" in df.columns: df.rename(columns={"Post_URL": "URL"}, inplace=True)
    if "Full_Post_Content" in df.columns: df.rename(columns={"Full_Post_Content": "Description"}, inplace=True)
    for col in ["Title", "Property_Details", "Price", "URL", "Description"]:
        if col not in df.columns: df[col] = ""
        df[col] = df[col].fillna('')

    final_rows = []
    total_rows = len(df)
    
    print("Starting processing...")
    for i, row in df.iterrows():
        print(f"--> Processing row {i+1}/{total_rows}...")
        
        full_text = normalize_text(f"{row.get('Title', '')} {row.get('Property_Details', '')} {row.get('Description', '')}")
        
        if not is_relevant_post(full_text):
            print(f"    - Skipping (Not a relevant property post)")
            continue
        
        user_message = build_user_message(row)
        llm_result = call_typhoon_analyzer(user_message)
        
        if not llm_result or not llm_result.get("extracted"):
            print(f"    - Skipping (LLM extraction failed)")
            continue
            
        output_row = build_row(row, llm_result)
        final_rows.append(output_row)
        print(f"    - Done. Amphoe: {output_row['อำเภอ']} | Project/Loc: {output_row['ทำเล'][:50]}...")

    print(f"\nProcessing complete. Found {len(final_rows)} relevant listings.")
    
    out_df = pd.DataFrame(final_rows)
    
    print(f"Saving to Excel file: {output_path}")
    out_df.to_excel(output_path, index=False, engine='openpyxl')
    print("Successfully saved to Excel.")

if __name__ == "__main__":
    input_path = r"C:\Users\kongl\Documents\GitHub\Real-Estate Listing Aggregator System\fazwaz_scraped_details.csv"
    process_and_save(input_path, OUTPUT_EXCEL_FILE)

Loading CSV from: C:\Users\kongl\Documents\GitHub\Real-Estate Listing Aggregator System\fazwaz_scraped_details.csv
Loaded 500 rows.
Starting processing...
--> Processing row 1/500...
    - Skipping (Not a relevant property post)
--> Processing row 2/500...
    - Skipping (Not a relevant property post)
--> Processing row 3/500...
    - Skipping (Not a relevant property post)
--> Processing row 4/500...
    - Done. Amphoe: สารภี | Project/Loc: สารภี, เชียงใหม่ | คลังสินค้า 4 ห้องนอน | ใกล้: ห้...
--> Processing row 5/500...
    - Done. Amphoe: เมืองเชียงใหม่ | Project/Loc: สันผีเสื้อ, เมืองเชียงใหม่, เชียงใหม่ | หมู่บ้านร้...
--> Processing row 6/500...
    - Done. Amphoe: แม่ริม | Project/Loc: Green Valley Condo (กรีน วัลเลย์ คอนโด) | แม่สา, แ...
--> Processing row 7/500...
    - Done. Amphoe: เมืองเชียงใหม่ | Project/Loc: โครงการ Glory Boutique Suites (กลอรี่ บูทีค สูท) |...
--> Processing row 8/500...
    - Done. Amphoe: เมืองเชียงใหม่ | Project/Loc: Karnkanok 3 Condo Jed Yod Greenery

In [17]:
import csv
import time
import re
from pathlib import Path
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

INPUT_CSV_FILE = Path(r"C:\Users\kongl\Documents\GitHub\Real-Estate Listing Aggregator System\Scraping\kaidee_listing_urls.csv")
OUTPUT_CSV_FILE = "kaidee_scraped_details.csv"
WAIT = 50

def scrape(driver, url):
    print(f"[Stage: Scrape] Open -> {url}")
    w = WebDriverWait(driver, WAIT)
    driver.get(url)

    btns = driver.find_elements(By.CSS_SELECTOR, "button[aria-label*='cookie i understand'], button[aria-label*='accept'], button:has(span[lang])")
    if btns:
        print("[Stage: Scrape] Click cookie/consent")
        driver.execute_script("arguments[0].click();", btns[0])
        time.sleep(0.3)

    print("[Stage: Scrape] Wait title")
    title_el = w.until(EC.presence_of_element_located((By.CSS_SELECTOR, "h1.sc-747m9u-7")))
    title_txt = title_el.text.strip()

    print("[Stage: Scrape] Try expand 'อ่านเพิ่มเติม' if present")
    read_more = driver.find_elements(By.XPATH, "//a[contains(normalize-space(),'อ่านเพิ่มเติม')]")
    if read_more:
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", read_more[0])
        driver.execute_script("arguments[0].click();", read_more[0])
        time.sleep(0.6)

    print("[Stage: Scrape] Wait price/attributes block")
    w.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.sc-12ljfib-0, div.sc-1w68tq4-0")))

    price_txt = ""
    price_candidates = driver.find_elements(By.CSS_SELECTOR, "span.sc-3tpgds-0.krrrAv")
    for el in price_candidates:
        t = el.text.strip()
        if re.search(r"\d", t):
            price_txt = t
            break
    if not price_txt:
        block = None
        blocks = driver.find_elements(By.CSS_SELECTOR, "div.sc-12ljfib-0, div.sc-1w68tq4-0")
        if blocks:
            block = blocks[0].text
        if block:
            m = re.search(r"([0-9][0-9,\.]{0,18})", block)
            if m:
                price_txt = m.group(1)
    if not price_txt:
        metas = driver.find_elements(By.CSS_SELECTOR, "meta[itemprop='price'], meta[property='product:price:amount']")
        if metas:
            v = metas[0].get_attribute("content") or ""
            v = v.strip()
            if v:
                price_txt = v

    print(f"[Stage: Scrape] Price parsed -> '{price_txt}'")

    print("[Stage: Scrape] Collect attributes (ul#has-attributes, include เนื้อที่)")
    attrs = []
    land_area_els = driver.find_elements(By.XPATH, "//ul[@id='has-attributes']//li[.//span[contains(normalize-space(),'เนื้อที่')]]//span//b")
    if land_area_els:
        v = land_area_els[0].text.strip()
        if v:
            attrs.append(f"เนื้อที่: {v}")
    li = driver.find_elements(By.CSS_SELECTOR, "ul#has-attributes li")
    for x in li:
        t = " ".join(x.text.split())
        if t:
            attrs.append(t)
    attrs_txt = " | ".join(dict.fromkeys([a for a in attrs if a]))

    print("[Stage: Scrape] Collect description (รายละเอียดสินค้า)")
    desc_root = driver.find_elements(By.CSS_SELECTOR, "div.sc-1kndlp1-0")
    if desc_root:
        paras = desc_root[0].find_elements(By.CSS_SELECTOR, "p.inner-text")
        desc_txt = "\n".join(p.text.strip() for p in paras if p.text.strip())
        masked = desc_root[0].find_elements(By.CSS_SELECTOR, "span.masked[data-value]")
        for m in masked:
            mv = (m.get_attribute("data-value") or "").strip()
            mt = (m.text or "").strip()
            if mv and mt:
                desc_txt = desc_txt.replace(mt, mv)
    else:
        desc_txt = ""

    print("[Stage: Scrape] Build Full_Post_Content")
    parts = []
    if price_txt:
        parts.append(f"ราคา: {price_txt}")
    if attrs_txt:
        parts.append(attrs_txt)
    if title_txt:
        parts.append(title_txt)
    if desc_txt:
        parts.append(desc_txt)
    full_text = "\n".join(parts).replace("อ่านเพิ่มเติม", "").replace("ดูน้อยลง", "").strip()
    print(f"[Stage: Scrape] Done -> {len(full_text)} chars")
    return {"Post_URL": url, "Full_Post_Content": full_text}

def main():
    print("[Stage: Init] Validate input CSV path")
    if not INPUT_CSV_FILE.exists():
        print(f"[Stage: Abort] Not found: {INPUT_CSV_FILE}")
        return

    print("[Stage: Init] Launch Chrome")
    options = uc.ChromeOptions()
    options.add_argument("--disable-notifications")
    options.add_argument("--start-maximized")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--no-sandbox")
    options.page_load_strategy = "eager"
    driver = uc.Chrome(options=options)
    driver.command_executor._client_config.timeout = 180
    driver.set_page_load_timeout(120)
    driver.set_script_timeout(120)

    print("[Stage: Load] Read URLs")
    with open(INPUT_CSV_FILE, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        next(reader, None)
        urls = [r[0].strip() for r in reader if r and r[0].strip()]
    print(f"[Stage: Load] Total URLs: {len(urls)}")

    print(f"[Stage: Save] Open output CSV for streaming append -> {OUTPUT_CSV_FILE}")
    header = ["Post_URL", "Full_Post_Content"]
    need_header = not Path(OUTPUT_CSV_FILE).exists() or Path(OUTPUT_CSV_FILE).stat().st_size == 0
    with open(OUTPUT_CSV_FILE, "a", newline="", encoding="utf-8") as f_out:
        w = csv.DictWriter(f_out, fieldnames=header)
        if need_header:
            w.writeheader()
            f_out.flush()
        for i, u in enumerate(urls, start=1):
            print(f"[Stage: Progress] {i}/{len(urls)}")
            row = scrape(driver, u)
            w.writerow(row)
            f_out.flush()
            time.sleep(0.8)

    print("[Stage: Teardown] Quit Chrome")
    driver.quit()
    print("[Stage: Done] kaidee full post content complete and CSV updated per URL")

if __name__ == "__main__":
    main()

[Stage: Init] Validate input CSV path
[Stage: Init] Launch Chrome
[Stage: Load] Read URLs
[Stage: Load] Total URLs: 784
[Stage: Save] Open output CSV for streaming append -> kaidee_scraped_details.csv
[Stage: Progress] 1/784
[Stage: Scrape] Open -> https://baan.kaidee.com/product-367612536
[Stage: Scrape] Wait title
[Stage: Scrape] Try expand 'อ่านเพิ่มเติม' if present
[Stage: Scrape] Wait price/attributes block
[Stage: Scrape] Price parsed -> '36,000'
[Stage: Scrape] Collect attributes (ul#has-attributes, include เนื้อที่)
[Stage: Scrape] Collect description (รายละเอียดสินค้า)
[Stage: Scrape] Build Full_Post_Content
[Stage: Scrape] Done -> 340 chars
[Stage: Progress] 2/784
[Stage: Scrape] Open -> https://baan.kaidee.com/product-367612538
[Stage: Scrape] Wait title
[Stage: Scrape] Try expand 'อ่านเพิ่มเติม' if present
[Stage: Scrape] Wait price/attributes block
[Stage: Scrape] Price parsed -> '850,000'
[Stage: Scrape] Collect attributes (ul#has-attributes, include เนื้อที่)
[Stage: Scr

TimeoutException: Message: timeout: Timed out receiving message from renderer: 42.623
  (Session info: chrome=142.0.7444.163)
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0xfe4103
	0xfe4144
	0xdee71d
	0xddec5a
	0xdde98d
	0xddc7fe
	0xddd3c7
	0xdea16e
	0xdfc095
	0xe01be6
	0xddda46
	0xdfbe27
	0xe7f14f
	0xe5c706
	0xe2da30
	0xe2ed54
	0x12557b4
	0x125098a
	0x100c392
	0xffc4c8
	0x100324d
	0xfec478
	0xfec63c
	0xfd67ca
	0x750e5d49
	0x76fed6db
	0x76fed661


In [18]:
import csv
import time
import re
from pathlib import Path
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

INPUT_CSV_FILE = Path(r"C:\Users\kongl\Documents\GitHub\Real-Estate Listing Aggregator System\Scraping\kaidee_listing_urls.csv")
OUTPUT_CSV_FILE = "kaidee_scraped_details.csv"
WAIT = 50
START_INDEX = 575

def scrape(driver, url):
    print(f"[Stage: Scrape] Open -> {url}")
    w = WebDriverWait(driver, WAIT)
    driver.get(url)

    btns = driver.find_elements(By.CSS_SELECTOR, "button[aria-label*='cookie i understand'], button[aria-label*='accept'], button:has(span[lang])")
    if btns:
        print("[Stage: Scrape] Click cookie/consent")
        driver.execute_script("arguments[0].click();", btns[0])
        time.sleep(0.3)

    print("[Stage: Scrape] Wait title")
    title_el = w.until(EC.presence_of_element_located((By.CSS_SELECTOR, "h1.sc-747m9u-7")))
    title_txt = title_el.text.strip()

    print("[Stage: Scrape] Try expand 'อ่านเพิ่มเติม' if present")
    read_more = driver.find_elements(By.XPATH, "//a[contains(normalize-space(),'อ่านเพิ่มเติม')]")
    if read_more:
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", read_more[0])
        driver.execute_script("arguments[0].click();", read_more[0])
        time.sleep(0.6)

    print("[Stage: Scrape] Wait price/attributes block")
    w.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.sc-12ljfib-0, div.sc-1w68tq4-0")))

    price_txt = ""
    price_candidates = driver.find_elements(By.CSS_SELECTOR, "span.sc-3tpgds-0.krrrAv")
    for el in price_candidates:
        t = el.text.strip()
        if re.search(r"\d", t):
            price_txt = t
            break
    if not price_txt:
        block = None
        blocks = driver.find_elements(By.CSS_SELECTOR, "div.sc-12ljfib-0, div.sc-1w68tq4-0")
        if blocks:
            block = blocks[0].text
        if block:
            m = re.search(r"([0-9][0-9,\.]{0,18})", block)
            if m:
                price_txt = m.group(1)
    if not price_txt:
        metas = driver.find_elements(By.CSS_SELECTOR, "meta[itemprop='price'], meta[property='product:price:amount']")
        if metas:
            v = metas[0].get_attribute("content") or ""
            v = v.strip()
            if v:
                price_txt = v

    print(f"[Stage: Scrape] Price parsed -> '{price_txt}'")

    print("[Stage: Scrape] Collect attributes (ul#has-attributes, include เนื้อที่)")
    attrs = []
    land_area_els = driver.find_elements(By.XPATH, "//ul[@id='has-attributes']//li[.//span[contains(normalize-space(),'เนื้อที่')]]//span//b")
    if land_area_els:
        v = land_area_els[0].text.strip()
        if v:
            attrs.append(f"เนื้อที่: {v}")
    li = driver.find_elements(By.CSS_SELECTOR, "ul#has-attributes li")
    for x in li:
        t = " ".join(x.text.split())
        if t:
            attrs.append(t)
    attrs_txt = " | ".join(dict.fromkeys([a for a in attrs if a]))

    print("[Stage: Scrape] Collect description (รายละเอียดสินค้า)")
    desc_root = driver.find_elements(By.CSS_SELECTOR, "div.sc-1kndlp1-0")
    if desc_root:
        paras = desc_root[0].find_elements(By.CSS_SELECTOR, "p.inner-text")
        desc_txt = "\n".join(p.text.strip() for p in paras if p.text.strip())
        masked = desc_root[0].find_elements(By.CSS_SELECTOR, "span.masked[data-value]")
        for m in masked:
            mv = (m.get_attribute("data-value") or "").strip()
            mt = (m.text or "").strip()
            if mv and mt:
                desc_txt = desc_txt.replace(mt, mv)
    else:
        desc_txt = ""

    print("[Stage: Scrape] Build Full_Post_Content")
    parts = []
    if price_txt:
        parts.append(f"ราคา: {price_txt}")
    if attrs_txt:
        parts.append(attrs_txt)
    if title_txt:
        parts.append(title_txt)
    if desc_txt:
        parts.append(desc_txt)
    full_text = "\n".join(parts).replace("อ่านเพิ่มเติม", "").replace("ดูน้อยลง", "").strip()
    print(f"[Stage: Scrape] Done -> {len(full_text)} chars")
    return {"Post_URL": url, "Full_Post_Content": full_text}

def main():
    print("[Stage: Init] Validate input CSV path")
    if not INPUT_CSV_FILE.exists():
        print(f"[Stage: Abort] Not found: {INPUT_CSV_FILE}")
        return

    print("[Stage: Init] Launch Chrome")
    options = uc.ChromeOptions()
    options.add_argument("--disable-notifications")
    options.add_argument("--start-maximized")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--no-sandbox")
    options.page_load_strategy = "eager"
    driver = uc.Chrome(options=options)
    driver.command_executor._client_config.timeout = 180
    driver.set_page_load_timeout(120)
    driver.set_script_timeout(120)

    print("[Stage: Load] Read URLs")
    with open(INPUT_CSV_FILE, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        next(reader, None)
        urls = [r[0].strip() for r in reader if r and r[0].strip()]
    print(f"[Stage: Load] Total URLs: {len(urls)}")

    urls_to_process = urls[START_INDEX-1:]
    print(f"[Stage: Resume] Start at URL index {START_INDEX}, remaining {len(urls_to_process)}")

    print(f"[Stage: Save] Open output CSV for streaming append -> {OUTPUT_CSV_FILE}")
    header = ["Post_URL", "Full_Post_Content"]
    need_header = not Path(OUTPUT_CSV_FILE).exists() or Path(OUTPUT_CSV_FILE).stat().st_size == 0
    with open(OUTPUT_CSV_FILE, "a", newline="", encoding="utf-8") as f_out:
        w = csv.DictWriter(f_out, fieldnames=header)
        if need_header:
            w.writeheader()
            f_out.flush()
        for i, u in enumerate(urls_to_process, start=START_INDEX):
            print(f"[Stage: Progress] {i}/{len(urls)}")
            row = scrape(driver, u)
            w.writerow(row)
            f_out.flush()
            time.sleep(0.8)

    print("[Stage: Teardown] Quit Chrome")
    driver.quit()
    print("[Stage: Done] kaidee full post content complete and CSV updated per URL")

if __name__ == "__main__":
    main()


[Stage: Init] Validate input CSV path
[Stage: Init] Launch Chrome
[Stage: Load] Read URLs
[Stage: Load] Total URLs: 784
[Stage: Resume] Start at URL index 575, remaining 210
[Stage: Save] Open output CSV for streaming append -> kaidee_scraped_details.csv
[Stage: Progress] 575/784
[Stage: Scrape] Open -> https://baan.kaidee.com/product-371048608
[Stage: Scrape] Wait title
[Stage: Scrape] Try expand 'อ่านเพิ่มเติม' if present
[Stage: Scrape] Wait price/attributes block
[Stage: Scrape] Price parsed -> '33,000'
[Stage: Scrape] Collect attributes (ul#has-attributes, include เนื้อที่)
[Stage: Scrape] Collect description (รายละเอียดสินค้า)
[Stage: Scrape] Build Full_Post_Content
[Stage: Scrape] Done -> 1799 chars
[Stage: Progress] 576/784
[Stage: Scrape] Open -> https://baan.kaidee.com/product-371048740
[Stage: Scrape] Wait title
[Stage: Scrape] Try expand 'อ่านเพิ่มเติม' if present
[Stage: Scrape] Wait price/attributes block
[Stage: Scrape] Price parsed -> '20,000'
[Stage: Scrape] Collect at